In [1]:
import warnings
from pathlib import Path

import numpy as np
import xarray as xr
import prism

from imagematerials import vehicles as vhc
from imagematerials import buildings as bld
from imagematerials.concepts import create_vehicle_graph, create_building_graph

from imagematerials.factory import ModelFactory, Sector
from imagematerials.maintenance import Maintenance
from imagematerials.model import (
    EndOfLife,
    GenericMainModel,
    GenericMaterials,
    GenericStocks,
    MaterialIntensities,
)
from imagematerials.util import (
    export_to_netcdf,
    import_from_netcdf,
    read_circular_economy_config,
    read_climate_policy_config,
    rebroadcast_prep_data,
)

from imagematerials.eol import preprocess_eol


## scenario config

In [2]:
base_dir = "../data/raw"
climate_policy_scenario_dir = Path(base_dir) / 'SSP2'
circular_economy_scenario_dirs = {"base": Path(base_dir) / 'circular_economy_scenarios' / 'base'}
prep_fp = Path("prep_vema.nc")
# note: configuration currently only works when no prep file is saved

## preprocessing 

In [3]:
# EoL
prep_eol = preprocess_eol("../../data/raw")

# Vehicles
def get_vema_prep():
    base_dir = Path("..", "data", "raw")
    prep_fp = Path("prep_vema.nc")
    if not prep_fp.is_file():
        climate_policy_scenario_dir = base_dir / 'SSP2'
        circular_economy_scenario_dirs = {"slow": base_dir / 'circular_economy_scenarios' / 'slow'}

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            climate_policy_config = read_climate_policy_config(climate_policy_scenario_dir)
            circular_economy_config = read_circular_economy_config(circular_economy_scenario_dirs)
            prep_data = vhc.preprocess(base_dir, climate_policy_config, circular_economy_config)

        export_to_netcdf(prep_data, prep_fp)

    prep_data = import_from_netcdf(prep_fp)

    target_materials = [
    "Aluminium", "Brick", "Cement", "Concrete", 
    "Copper", "Glass", "Steel", "Wood"
]
    
    prep_data['battery_materials'] = prep_data['battery_materials'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb','Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )
    prep_data['battery_materials'] = prep_data['battery_materials'].reindex(material=target_materials)
    prep_data['material_fractions'] = prep_data['material_fractions'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb','Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )
    prep_data['material_fractions'] = prep_data['material_fractions'].reindex(material=target_materials)
    prep_data['maintenance_material_fractions'] = prep_data['maintenance_material_fractions'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb','Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )
    prep_data['maintenance_material_fractions'] = prep_data['maintenance_material_fractions'].reindex(material=target_materials)
    
    share_coords = set()
    for cur_type in prep_data["shares"].Type.values:
        share_coords.add(cur_type.split(" - ")[0])
    output_coords_type = [x for x in prep_data["stocks"].Type.values if x not in share_coords] + list(prep_data["shares"].coords["Type"].values)
    prep_data.pop("shares")
    knowledge_graph = create_vehicle_graph()
    new_prep_data = rebroadcast_prep_data(prep_data, knowledge_graph, dim="Type", output_coords=output_coords_type)
    region_coords = np.sort(prep_data["stocks"].coords["Region"].values.astype(int)).astype(str)[:-2]
    new_prep_data = rebroadcast_prep_data(new_prep_data, knowledge_graph, dim="Region", output_coords=region_coords)
    new_prep_data["knowledge_graph"] = knowledge_graph
    new_prep_data["weights"] = new_prep_data.pop("vehicle_weights")
    new_prep_data["shares"] = None
    sec_vhc = Sector("vhc", new_prep_data)
    return sec_vhc
    

# Buildings

def get_buildings_prep():
    base_dir = Path("..", "IMAGE-Mat_old_version", "IMAGE-Mat", "BUMA")
    prep_fp = Path("prep_buildings.nc")
    if not prep_fp.is_file():
        prep_data = bld.preprocess(base_dir)
        export_to_netcdf(prep_data, prep_fp)
    else:
        prep_data = import_from_netcdf(prep_fp)
    new_prep_data = {k: v for k, v in prep_data.items()}
    new_prep_data["knowledge_graph"] = create_building_graph()
    new_prep_data["shares"] = None
    sec_bld = Sector("bld", new_prep_data)
    return sec_bld

In [4]:
# tests
# prep_data = import_from_netcdf(prep_fp)
# # prep_data['material_fractions'] = prep_data['material_fractions'].assign_coords(
# #     material=['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb',
# #               'Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'])
# # prep_data['lifetimes'].material.values
# prep_data.keys()



# # prep_data['material_fractions'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb',
# #       'Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )
# #.assign_coords('material' = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb',
# #       'Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'])
# prep_data['maintenance_material_fractions']
# prep_data['battery_materials'] = prep_data['battery_materials'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb','Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )

# prep_data['battery_materials'].sel(material = "Copper") # deu certo!!!


In [4]:
sec_vhc = get_vema_prep() 
sec_bld = get_buildings_prep()
eol_sector = Sector(name="eol", data = prep_eol)

In [ ]:

sec_bld.all_data

{'lifetimes': {'weibull': <xarray.DataArray 'weibull' (Time: 340, Region: 26, Type: 12, ScipyParam: 2)> Size: 2MB
  array([[[[  1.44      ,  49.57      ],
           [  1.44      ,  49.57      ],
           [  1.44      ,  49.57      ],
           ...,
           [  1.96820464,  57.52862846],
           [  1.96820464,  57.52862846],
           [  1.96820464,  57.52862846]],
  
          [[  1.44      ,  49.57      ],
           [  1.44      ,  49.57      ],
           [  1.44      ,  49.57      ],
           ...,
           [  4.16343417,  85.18683893],
           [  4.16343417,  85.18683893],
           [  4.16343417,  85.18683893]],
  
          [[  1.44      ,  49.57      ],
           [  1.44      ,  49.57      ],
           [  1.44      ,  49.57      ],
           ...,
  ...
           ...,
           [  1.96820464, 122.2       ],
           [  1.96820464, 122.2       ],
           [  1.96820464, 122.2       ]],
  
          [[  1.44      ,  94.18      ],
           [  1.44      ,

In [5]:
sectors = [sec_vhc, sec_bld,eol_sector]

## define timeline

In [6]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

## build factory for eol

In [7]:
REGION = prism.Dimension("Region")
MATERIAL_TYPE = prism.Dimension("material")


factory = ModelFactory(sectors, complete_timeline)
factory.add(GenericStocks, ["vhc","bld"])
factory.add(GenericMaterials, "vhc")
factory.add(MaterialIntensities,"bld")

factory.add(EndOfLife, "eol", input_sources={
    "outflow_by_cohort_materials": "vhc",
    "outflow_by_cohort_materials": "bld",
    "collection": "eol",
    "reuse": "eol",
    "recycling": "eol"}
)

model = factory.finish()

In [8]:
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)

In [9]:
model.vhc['outflow_by_cohort_materials']

TimeVariable(
  timeline=Timeline(start=1960, end=2060, stepsize=1),
  unit=count,
  dims=
	Dimension(label="Region", coords=[np.str_('1'), np.str_('2'), np.str_('3'),
	  np.str_('4'), np.str_('5'), np.str_('6'), np.str_('7'), np.str_('8'),
	  np.str_('9'), np.str_('10'), np.str_('11'), np.str_('12'),
	  np.str_('13'), np.str_('14'), np.str_('15'), np.str_('16'),
	  np.str_('17'), np.str_('18'), np.str_('19'), np.str_('20'),
	  np.str_('21'), np.str_('22'), np.str_('23'), np.str_('24'),
	  np.str_('25'), np.str_('26')])
        Dimension(label="Type",
	  coords=[np.str_('Bikes'), np.str_('Freight Planes'), np.str_('Freight
	  Trains'), np.str_('High Speed Trains'), np.str_('Inland Ships'),
	  np.str_('Large Ships'), np.str_('Medium Ships'), np.str_('Passenger
	  Planes'), np.str_('Small Ships'), np.str_('Trains'), np.str_('Very Large
	  Ships'), np.str_('Cars - BEV'), np.str_('Cars - FCV'), np.str_('Cars -
	  HEV'), np.str_('Cars - ICE'), np.str_('Cars - PHEV'), np.str_('Cars -
	  Trol

In [9]:
model.eol['sum_outflow'][2020]

Magnitude,[[[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] ... [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]]]
Units,count


In [10]:
model.eol['sum_outflow'][2020]

Magnitude,[[[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] ... [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]] [[nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] ... [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan] [nan nan nan ... nan nan nan]]]
Units,count
